# AML Gremlin Showcase on HugeGraph

 This notebook implements anti-money-laundering (AML) analysis as executable Gremlin traversals in HugeGraph, using the following schema:

- vertex label: `account`
- edge label: `transfer`
- account properties: `bank`, `acct`
- transfer properties: `ts`, `amount_paid`, `amount_recv`, `pay_currency`,
  `recv_currency`, `pay_format`, `is_laundering`, `row`

The focus here is executing and observing Gremlin query behavior in HugeGraph.
To stay stable on large datasets, queries are intentionally scoped with `limit(...)`
and seed vertices.


## Step 0 - Configuration

- Start the cloud-storage stack first.
- Put `LI-Large_Trans.csv` and `HI-Large_accounts.csv` under `docker/cloud-storage/notebooks/data`
  (or another candidate path shown in config); this notebook can load the data in Step 2.


## Step 1 - Helpers


In [ ]:
# Step 1 can run standalone even if Step 0/config cells were not executed yet.
import json
import time
import requests

HG_HOST = "127.0.0.1"  # HugeGraph server host.
HG_PORT = 8080  # HugeGraph server port.
GRAPH = "hugegraph"  # Graph name used by this notebook.

BASE = f"http://{HG_HOST}:{HG_PORT}/graphs/{GRAPH}"  # Graph REST base endpoint.
GREMLIN = f"http://{HG_HOST}:{HG_PORT}/gremlin"  # Gremlin HTTP endpoint.

GREMLIN_EVAL_TIMEOUT_MS = 180_000  # Default Gremlin evaluation timeout per query.
ROW_PRINT_LIMIT = 20  # Max rows printed by show().
ALIAS_STABILITY_TIMEOUT_SEC = 180  # Max seconds to wait for a stable alias binding.
ALIAS_STABILITY_POLL_SEC = 2  # Seconds between alias-binding probes.
ALIAS_STABILITY_CONSECUTIVE_SUCCESSES = 3  # Successes required before pinning alias.
ALIAS_STABILITY_QUERY = "g.V().limit(1).count()"  # Probe query used for alias checks.
GREMLIN_PINNED_ALIAS = None  # Optional fixed alias (None enables dynamic probing/fallback).

session = requests.Session()
session.headers.update({"Content-Type": "application/json"})

GREMLIN_ALIAS_CANDIDATES = [
    f"__g_DEFAULT-{GRAPH}",
    f"__g_{GRAPH}",
    GRAPH,
]


def ordered_alias_candidates():
    if GREMLIN_PINNED_ALIAS in GREMLIN_ALIAS_CANDIDATES:
        return [
            GREMLIN_PINNED_ALIAS,
            *[a for a in GREMLIN_ALIAS_CANDIDATES if a != GREMLIN_PINNED_ALIAS],
        ]
    return list(GREMLIN_ALIAS_CANDIDATES)

TRANSIENT_GREMLIN_ERRORS = (
    "refCnt: 0, decrement: 1",
    "SocketTimeoutException",
    "Read timed out",
    "ProcessingException",
    "RejectedExecutionException",
    "connect timed out",
    "connection reset",
    "connection refused",
)

ALIAS_FALLBACK_ERRORS = (
    "No such property: g",
    "Could not rebind [g]",
    "TraversalSource global bindings",
)


def _contains_error_token(message, tokens):
    text = str(message or "").lower()
    return any(token.lower() in text for token in tokens)


def is_transient_gremlin_error(message):
    return _contains_error_token(message, TRANSIENT_GREMLIN_ERRORS)


def needs_alias_fallback(message):
    return _contains_error_token(message, ALIAS_FALLBACK_ERRORS)


def is_timeout_gremlin_error(message):
    return _contains_error_token(
        message,
        (
            "timeoutexception",
            "evaluation exceeded",
            "evaluationtimeout",
        ),
    )


def _check(r):
    if r.status_code >= 400:
        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:800]}")
    return r


def _post_gremlin(body, retries=2, timeout=240):
    last_err = ""
    for attempt in range(retries + 1):
        try:
            r = session.post(GREMLIN, data=json.dumps(body), timeout=timeout)
        except requests.RequestException as e:
            err = str(e)
            last_err = err
            if is_transient_gremlin_error(err) and attempt < retries:
                time.sleep(1 + attempt)
                continue
            return None, err

        if r.status_code < 400:
            return r.json()["result"]["data"], ""

        err = r.text[:800]
        last_err = err
        if is_transient_gremlin_error(err) and attempt < retries:
            time.sleep(1 + attempt)
            continue
        return None, err
    return None, last_err


def gremlin(query, bindings=None, eval_timeout_ms=GREMLIN_EVAL_TIMEOUT_MS):
    body = {"gremlin": query, "language": "gremlin-groovy"}
    if bindings:
        body["bindings"] = bindings
    if eval_timeout_ms is not None:
        body["evaluationTimeout"] = int(eval_timeout_ms)

    if GREMLIN_PINNED_ALIAS:
        body_with_alias = dict(body)
        body_with_alias["aliases"] = {"g": GREMLIN_PINNED_ALIAS}
        result, err = _post_gremlin(body_with_alias)
    else:
        result, err = _post_gremlin(body)
    if result is not None:
        return result

    # If a pinned alias executed but failed for a non-alias reason (for example timeout),
    # surface that root cause directly instead of masking it with alias-rebind noise.
    if GREMLIN_PINNED_ALIAS and not needs_alias_fallback(err):
        raise RuntimeError(
            f"Gremlin query failed with pinned alias '{GREMLIN_PINNED_ALIAS}': {err}"
        )

    if not needs_alias_fallback(err) and not is_transient_gremlin_error(err):
        raise RuntimeError(f"Gremlin query failed: {err}")

    last_err = err
    last_timeout_err = err if is_timeout_gremlin_error(err) else ""
    alias_rounds = 2
    for round_idx in range(alias_rounds):
        for alias in ordered_alias_candidates():
            body_with_alias = dict(body)
            body_with_alias["aliases"] = {"g": alias}
            result, alias_err = _post_gremlin(body_with_alias)
            if result is not None:
                return result

            last_err = alias_err
            if is_timeout_gremlin_error(alias_err):
                last_timeout_err = alias_err
            if needs_alias_fallback(alias_err) or is_transient_gremlin_error(alias_err):
                continue
            raise RuntimeError(f"Gremlin query failed with alias '{alias}': {alias_err}")

        if round_idx + 1 < alias_rounds and is_transient_gremlin_error(last_err):
            time.sleep(1 + round_idx)

    if last_timeout_err:
        raise RuntimeError(
            "Gremlin query timed out after alias fallback. Tried aliases "
            f"{ordered_alias_candidates()}. Last timeout: {last_timeout_err}"
        )

    raise RuntimeError(
        "Gremlin query failed after alias fallback. Tried aliases "
        f"{ordered_alias_candidates()}. Last error: {last_err}"
    )


def server_up():
    try:
        return session.get(f"http://{HG_HOST}:{HG_PORT}/graphs", timeout=5).status_code == 200
    except Exception:
        return False


def _probe_alias(alias, query=ALIAS_STABILITY_QUERY, eval_timeout_ms=10_000):
    body = {
        "gremlin": query,
        "language": "gremlin-groovy",
        "aliases": {"g": alias},
        "evaluationTimeout": int(eval_timeout_ms),
    }
    result, err = _post_gremlin(body, retries=1, timeout=20)
    return result is not None, err


def wait_for_gremlin_alias_binding(
    timeout_sec=ALIAS_STABILITY_TIMEOUT_SEC,
    poll_sec=ALIAS_STABILITY_POLL_SEC,
    required_consecutive=ALIAS_STABILITY_CONSECUTIVE_SUCCESSES,
):
    global GREMLIN_PINNED_ALIAS

    deadline = time.time() + int(timeout_sec)
    stable_alias = None
    consecutive = 0
    checks = 0
    last_errors = {}

    while time.time() < deadline:
        checks += 1
        winner = None

        for alias in ordered_alias_candidates():
            ok, err = _probe_alias(alias)
            if ok:
                winner = alias
                break
            last_errors[alias] = err

        if winner:
            if winner == stable_alias:
                consecutive += 1
            else:
                stable_alias = winner
                consecutive = 1

            print(
                f"Alias probe {checks}: using '{winner}' "
                f"({consecutive}/{int(required_consecutive)})"
            )

            if consecutive >= int(required_consecutive):
                GREMLIN_PINNED_ALIAS = winner
                GREMLIN_ALIAS_CANDIDATES[:] = [
                    winner,
                    *[a for a in GREMLIN_ALIAS_CANDIDATES if a != winner],
                ]
                print(f"Gremlin alias binding is stable. Pinned alias: {winner}")
                return winner
        else:
            stable_alias = None
            consecutive = 0
            if checks % 5 == 0:
                preview = "; ".join(
                    f"{a}: {str(e)[:120]}" for a, e in list(last_errors.items())[:3]
                )
                print(
                    "Alias probe pending: no alias succeeded yet; "
                    f"retrying in {int(poll_sec)}s. Last errors: {preview}"
                )

        time.sleep(int(poll_sec))

    raise TimeoutError(
        "Timed out waiting for stable Gremlin alias binding. "
        f"Tried aliases {ordered_alias_candidates()} for {int(timeout_sec)}s"
    )


def _format_row(row, max_len=260):
    try:
        text = json.dumps(row, ensure_ascii=True)
    except Exception:
        text = str(row)
    return text if len(text) <= max_len else text[: max_len - 3] + "..."


def show(query, title="", bindings=None, eval_timeout_ms=GREMLIN_EVAL_TIMEOUT_MS,
         row_limit=ROW_PRINT_LIMIT):
    if title:
        print(f"\n=== {title} ===")
    print("Gremlin:", query)
    try:
        rows = gremlin(query, bindings=bindings, eval_timeout_ms=eval_timeout_ms)
    except Exception as e:
        print("FAILED:", e)
        return []

    if not isinstance(rows, list):
        print("result:", rows)
        return rows

    print(f"rows={len(rows)}")
    for row in rows[: int(row_limit)]:
        print(" ", _format_row(row))
    if len(rows) > int(row_limit):
        print(f" ... ({len(rows) - int(row_limit)} more rows)")
    return rows



## Step 1.5 - Preflight alias-binding gate

Blocks execution until one Gremlin traversal-source alias is repeatedly successful.
This reduces intermittent `No such property: g` failures later in the notebook.


In [ ]:
wait_for_gremlin_alias_binding()


In [ ]:
import json
import time
import csv
import shutil
import subprocess
from pathlib import Path
import requests

HG_HOST = "127.0.0.1"  # HugeGraph server host.
HG_PORT = 8080  # HugeGraph server port.
GRAPH = "hugegraph"  # Graph name used by this notebook.

BASE = f"http://{HG_HOST}:{HG_PORT}/graphs/{GRAPH}"  # Graph REST base endpoint.
GREMLIN = f"http://{HG_HOST}:{HG_PORT}/gremlin"  # Gremlin HTTP endpoint.

GREMLIN_EVAL_TIMEOUT_MS = 180_000  # Default Gremlin evaluation timeout per query.
ROW_PRINT_LIMIT = 20  # Max rows printed by show().
ALIAS_STABILITY_TIMEOUT_SEC = 180  # Max seconds to wait for a stable alias binding.
ALIAS_STABILITY_POLL_SEC = 2  # Seconds between alias-binding probes.
ALIAS_STABILITY_CONSECUTIVE_SUCCESSES = 3  # Successes required before pinning alias.
ALIAS_STABILITY_QUERY = "g.V().limit(1).count()"  # Probe query used for alias checks.
# Set this to a specific alias string to force pinning (for example '__g_DEFAULT-hugegraph').
# Keep None to allow dynamic alias fallback/probing.
GREMLIN_PINNED_ALIAS = None

# Local AML CSV inputs (auto-detect common locations).
TRANS_CSV_NAME = "LI-Large_Trans.csv"  # Primary AML transactions CSV file name.
ACCOUNTS_CSV_NAME = "HI-Large_accounts.csv"  # Optional accounts CSV file name.
DATA_DIR_CANDIDATES = [
    Path("./data"),
    Path("./docker/cloud-storage/notebooks/data"),
    Path("./notebooks/data"),
    Path("../notebooks/data"),
    Path("."),
]
# Pick the first candidate directory that already contains the transactions CSV.
DATA_DIR = next(
    (d for d in DATA_DIR_CANDIDATES if (d / TRANS_CSV_NAME).exists()),
    DATA_DIR_CANDIDATES[0],
)
TRANS_CSV = DATA_DIR / TRANS_CSV_NAME  # Resolved transactions CSV path.
ACCOUNTS_CSV = DATA_DIR / ACCOUNTS_CSV_NAME  # Resolved accounts CSV path.

# Loader controls.
LOAD_ENABLED = True  # Master switch for Step 2 schema/data load.
LOAD_IF_GRAPH_EMPTY = True  # Load only if graph has no transfer data.
FORCE_RELOAD = False  # Force load even when data already exists.
CLEAR_GRAPH_BEFORE_LOAD = False  # Clear graph data before loading.
LOAD_ROWS = 5000000  # Max transaction rows to load; set None to load full file.
VERTEX_BATCH = 2000  # Vertex REST batch size for insert calls.
EDGE_BATCH = 2000  # Edge REST batch size for insert calls.
LOAD_PROGRESS_EVERY_ROWS = 100_000  # Print loader progress every N rows.
MAX_SEEN_ACCOUNTS = 4_000_000  # Soft cap for in-memory account dedup set.
SCHEMA_HTTP_TIMEOUT_SEC = 20  # Timeout for schema REST requests.
SCHEMA_TASK_TIMEOUT_SEC = 120  # Max wait time for async schema tasks.
SCHEMA_TASK_POLL_SEC = 2  # Poll interval for schema task status checks.
KAGGLE_AUTO_DOWNLOAD = True  # Download AML CSVs if missing.
KAGGLE_DATASET = "ealtman2019/ibm-transactions-for-anti-money-laundering-aml"  # Kaggle dataset slug.
KAGGLE_TOKEN_PATH = Path.home() / ".kaggle" / "kaggle.json"  # Kaggle API token path.

# Scoped traversal knobs (edit values here to override defaults for this notebook run).

# Deep-traversal safety controls (used by Step 15d scoped 6-hop query).
HOP6_OUT_LIMIT = 1200  # Max outgoing edges expanded per hop in 6-hop scoped traversal.
HOP6_RESULT_LIMIT = 50000  # Cap on unique 6-hop results after dedup.

# Scoped query safety controls (tune for speed vs coverage).
ROOT_COUNT_SCOPE_LIMIT = 1000  # Scope for root sample counts (Step 1a/1b).
HASNOT_SCOPE_LIMIT = 10000  # Scope for hasNot() check (Step 2d).
GROUPCOUNT_ACCOUNT_SCOPE_LIMIT = 300  # Vertex scope before currency groupCount (Step 5e).
GROUPCOUNT_EDGE_SCOPE_LIMIT = 5000  # Edge scope after account expansion (Step 5e).
RANGE_SCOPE_LIMIT = 2000  # Scope for ordering/range paging examples (Step 7a/7b/7c).
DEDUP_SCOPE_LIMIT = 200  # Scope for neighbor dedup count (Step 7d).
RANKING_SCOPE_LIMIT = 3000  # Scope for flagged account ranking (Step 14a).
HOP8_OUT_LIMIT = 1000  # Max outgoing edges expanded per hop in 8-hop traversal (Step 15b).
HOP8_RESULT_LIMIT = 50000  # Cap on unique 8-hop results after dedup (Step 15b).
ALIAS_PAIR_SCOPE_LIMIT = 200  # Scope for as/select pair sampling (Step 8a).
FLAGGED_PROJECT_SCOPE_LIMIT = 1000  # Scope for flagged_out projection ranking (Step 8b).
COMPOUND_WHERE_SCOPE_LIMIT = 1000  # Scope for compound where() examples (Step 9).
BANK_GROUP_SCOPE_LIMIT = 5000  # Scope for bank-level account groupCount (Step 10a).

print("Graph REST:", BASE)
print("Gremlin endpoint:", GREMLIN)
print("Data directory:", DATA_DIR)
print("Transactions CSV:", TRANS_CSV)
print("Accounts CSV:", ACCOUNTS_CSV)
print("Load enabled:", LOAD_ENABLED)
print("Load rows:", "full file" if LOAD_ROWS is None else int(LOAD_ROWS))
print("Kaggle auto download:", KAGGLE_AUTO_DOWNLOAD)
print("Kaggle dataset:", KAGGLE_DATASET)
print("Step 15d HOP6_OUT_LIMIT:", int(HOP6_OUT_LIMIT))
print("Step 15d HOP6_RESULT_LIMIT:", int(HOP6_RESULT_LIMIT))
print(
    "Scoped safety limits:",
    {
        "ROOT_COUNT_SCOPE_LIMIT": int(ROOT_COUNT_SCOPE_LIMIT),
        "HASNOT_SCOPE_LIMIT": int(HASNOT_SCOPE_LIMIT),
        "GROUPCOUNT_ACCOUNT_SCOPE_LIMIT": int(GROUPCOUNT_ACCOUNT_SCOPE_LIMIT),
        "GROUPCOUNT_EDGE_SCOPE_LIMIT": int(GROUPCOUNT_EDGE_SCOPE_LIMIT),
        "RANGE_SCOPE_LIMIT": int(RANGE_SCOPE_LIMIT),
        "DEDUP_SCOPE_LIMIT": int(DEDUP_SCOPE_LIMIT),
        "RANKING_SCOPE_LIMIT": int(RANKING_SCOPE_LIMIT),
        "HOP8_OUT_LIMIT": int(HOP8_OUT_LIMIT),
        "HOP8_RESULT_LIMIT": int(HOP8_RESULT_LIMIT),
        "ALIAS_PAIR_SCOPE_LIMIT": int(ALIAS_PAIR_SCOPE_LIMIT),
        "FLAGGED_PROJECT_SCOPE_LIMIT": int(FLAGGED_PROJECT_SCOPE_LIMIT),
        "COMPOUND_WHERE_SCOPE_LIMIT": int(COMPOUND_WHERE_SCOPE_LIMIT),
        "BANK_GROUP_SCOPE_LIMIT": int(BANK_GROUP_SCOPE_LIMIT),
    },
)


## Step 1.6 - CSV preflight (required for end-to-end runs)

This step always checks whether `LI-Large_Trans.csv` and
`HI-Large_accounts.csv` already exist.

- If files exist, it only reports status (no re-download).
- If files are missing and `KAGGLE_AUTO_DOWNLOAD=True`, it downloads from Kaggle.
- If files are missing and `KAGGLE_AUTO_DOWNLOAD=False`, Step 2 fails fast with a clear error.

Requirements:
- `kaggle` CLI installed in the kernel environment
- API token at `~/.kaggle/kaggle.json`


In [ ]:
def _file_size_mb(path):
    return f"{path.stat().st_size / 1e6:.1f} MB"


def show_aml_csv_status():
    for csv_path in (TRANS_CSV, ACCOUNTS_CSV):
        status = "found" if csv_path.exists() else "MISSING"
        size = _file_size_mb(csv_path) if csv_path.exists() else "-"
        print(f"  [{status}] {csv_path} ({size})")


def download_aml_csvs_from_kaggle(dataset=KAGGLE_DATASET, data_dir=DATA_DIR, files=None):
    files = list(files or [TRANS_CSV_NAME, ACCOUNTS_CSV_NAME])
    kaggle_bin = shutil.which("kaggle")
    if kaggle_bin is None:
        raise RuntimeError(
            "kaggle CLI not found. Install with: pip install kaggle"
        )
    if not KAGGLE_TOKEN_PATH.exists():
        raise RuntimeError(
            f"Kaggle token not found at {KAGGLE_TOKEN_PATH}. "
            "Create it from your Kaggle account API page."
        )

    data_dir.mkdir(parents=True, exist_ok=True)

    for fname in files:
        target = data_dir / fname
        if target.exists():
            print(f"  [found] {target} ({_file_size_mb(target)})")
            continue

        cmd = [
            kaggle_bin, "datasets", "download", dataset,
            "-f", fname,
            "-p", str(data_dir),
            "--unzip",
        ]
        print("Running:", " ".join(cmd))
        try:
            # Keep Kaggle stdout/stderr attached so notebook shows live download progress
            subprocess.run(cmd, check=True, text=True)
        except subprocess.CalledProcessError as e:
            err = (e.stderr or e.stdout or str(e)).strip()
            raise RuntimeError(
                f"Kaggle download failed for {fname}: {err[-500:]}"
            ) from e

        if not target.exists():
            raise RuntimeError(
                f"Kaggle command completed but expected file is missing: {target}"
            )
        print(f"  [downloaded] {target} ({_file_size_mb(target)})")


def ensure_aml_csvs(download_if_missing=KAGGLE_AUTO_DOWNLOAD):
    missing = [p for p in (TRANS_CSV, ACCOUNTS_CSV) if not p.exists()]
    if missing and download_if_missing:
        print("Missing AML CSV files -> attempting Kaggle download...")
        download_aml_csvs_from_kaggle()
    show_aml_csv_status()


ensure_aml_csvs(download_if_missing=KAGGLE_AUTO_DOWNLOAD)


## Step 2 - Schema and idempotent data load

Creates the AML schema if needed and loads `LI-Large_Trans.csv`
into HugeGraph only when required by loader settings.

With defaults, if transfer data already exists in the graph,
the load is skipped. Set `FORCE_RELOAD=True` to load again.


In [ ]:
def _schema_get(url, action):
    try:
        return session.get(url, timeout=SCHEMA_HTTP_TIMEOUT_SEC)
    except requests.RequestException as e:
        raise RuntimeError(
            f"[schema] {action} failed "
            f"(GET timeout={int(SCHEMA_HTTP_TIMEOUT_SEC)}s): {e}"
        ) from e


def _schema_post(url, body, action):
    try:
        return session.post(
            url,
            data=json.dumps(body),
            timeout=SCHEMA_HTTP_TIMEOUT_SEC,
        )
    except requests.RequestException as e:
        raise RuntimeError(
            f"[schema] {action} failed "
            f"(POST timeout={int(SCHEMA_HTTP_TIMEOUT_SEC)}s): {e}"
        ) from e


def create_property_key(name, data_type="TEXT", cardinality="SINGLE"):
    print(f"[schema] pk {name}")
    if _schema_get(f"{BASE}/schema/propertykeys/{name}", f"check pk {name}").status_code == 200:
        print(f"[schema] pk {name}: exists")
        return
    _check(_schema_post(
        f"{BASE}/schema/propertykeys",
        {"name": name, "data_type": data_type, "cardinality": cardinality},
        f"create pk {name}",
    ))
    print(f"[schema] pk {name}: created")


def create_vertex_label(name, props, id_strategy="AUTOMATIC", primary_keys=None):
    print(f"[schema] vl {name}")
    if _schema_get(f"{BASE}/schema/vertexlabels/{name}", f"check vl {name}").status_code == 200:
        print(f"[schema] vl {name}: exists")
        return
    body = {
        "name": name,
        "id_strategy": id_strategy,
        "properties": props,
        "nullable_keys": [p for p in props if p not in (primary_keys or [])],
    }
    if primary_keys:
        body["primary_keys"] = primary_keys
    _check(_schema_post(f"{BASE}/schema/vertexlabels", body, f"create vl {name}"))
    print(f"[schema] vl {name}: created")


def create_edge_label(name, source, target, props, frequency="SINGLE", sort_keys=None):
    print(f"[schema] el {name}")
    if _schema_get(f"{BASE}/schema/edgelabels/{name}", f"check el {name}").status_code == 200:
        print(f"[schema] el {name}: exists")
        return
    body = {
        "name": name,
        "source_label": source,
        "target_label": target,
        "properties": props,
        "frequency": frequency,
        "nullable_keys": [p for p in props if p not in (sort_keys or [])],
    }
    if sort_keys:
        body["sort_keys"] = sort_keys
    _check(_schema_post(f"{BASE}/schema/edgelabels", body, f"create el {name}"))
    print(f"[schema] el {name}: created")


def wait_task(task_id, timeout_sec=SCHEMA_TASK_TIMEOUT_SEC):
    timeout_sec = int(timeout_sec)
    deadline = time.time() + timeout_sec
    last_status = "UNKNOWN"
    last_reported_status = None
    unknown_count = 0
    print(f"[schema] index task id: {task_id} (timeout={timeout_sec}s)")
    while time.time() < deadline:
        payload = _check(
            _schema_get(f"{BASE}/tasks/{task_id}", f"check task {task_id}")
        ).json()
        # HugeGraph returns task fields like task_status/task_progress in task APIs.
        task = payload.get("task") if isinstance(payload, dict) else None
        if not isinstance(task, dict):
            task = payload if isinstance(payload, dict) else {}

        raw_status = (
            task.get("task_status") or
            task.get("status") or
            payload.get("task_status") if isinstance(payload, dict) else ""
        )
        progress = task.get("task_progress") or task.get("progress") or ""
        last_status = str(raw_status or "").upper() or "UNKNOWN"

        if last_status == "UNKNOWN":
            unknown_count += 1
            if unknown_count >= 3:
                keys = sorted(task.keys()) if isinstance(task, dict) else []
                raise RuntimeError(
                    f"[schema] task {task_id} response missing status fields after {unknown_count} polls. "
                    f"Task keys={keys}, payload={payload}"
                )
        else:
            unknown_count = 0

        if last_status != last_reported_status:
            suffix = f" progress={progress}" if progress != "" else ""
            print(f"[schema] task {task_id}: {last_status}{suffix}")
            last_reported_status = last_status

        if last_status in {"SUCCESS", "FAILED", "CANCELED", "CANCELLED"}:
            if last_status != "SUCCESS":
                raise RuntimeError(f"[schema] task {task_id} failed with status={last_status}: {task}")
            return
        time.sleep(int(SCHEMA_TASK_POLL_SEC))
    raise TimeoutError(
        f"[schema] task {task_id} timed out after {timeout_sec}s "
        f"(last status={last_status})"
    )


def create_index_label(name, base_type, base_value, index_type, fields):
    print(f"[schema] il {name}")
    if _schema_get(f"{BASE}/schema/indexlabels/{name}", f"check il {name}").status_code == 200:
        print(f"[schema] il {name}: exists")
        return
    body = {
        "name": name,
        "base_type": base_type,
        "base_value": base_value,
        "index_type": index_type,
        "fields": fields,
    }
    resp = _check(
        _schema_post(f"{BASE}/schema/indexlabels", body, f"create il {name}")
    ).json()
    task_id = resp.get("task_id")
    if task_id not in (None, 0, "0"):
        wait_task(task_id)
    print(f"[schema] il {name}: created")


def ensure_aml_schema():
    create_property_key("bank", "TEXT")
    create_property_key("acct", "TEXT")
    create_property_key("ts", "TEXT")
    create_property_key("amount_paid", "DOUBLE")
    create_property_key("amount_recv", "DOUBLE")
    create_property_key("pay_currency", "TEXT")
    create_property_key("recv_currency", "TEXT")
    create_property_key("pay_format", "TEXT")
    create_property_key("is_laundering", "INT")
    create_property_key("row", "LONG")

    create_vertex_label("account", ["bank", "acct"], id_strategy="CUSTOMIZE_STRING")
    create_edge_label(
        "transfer", "account", "account",
        [
            "ts", "amount_paid", "amount_recv", "pay_currency", "recv_currency",
            "pay_format", "is_laundering", "row",
        ],
        frequency="MULTIPLE", sort_keys=["row"],
    )
    create_index_label(
        "transfer_by_is_laundering",
        "EDGE_LABEL",
        "transfer",
        "SECONDARY",
        ["is_laundering"],
    )
    create_index_label(
        "account_by_bank",
        "VERTEX_LABEL",
        "account",
        "SECONDARY",
        ["bank"],
    )


def is_retriable_write_error(message):
    text = str(message or "").lower()
    return (
        is_transient_gremlin_error(text) or
        "cluster is not ready" in text or
        "active stores" in text or
        "partition_fault_type_unknown" in text
    )


def post_batch_with_retry(url, payload, label, attempts=12, base_sleep=2, max_sleep=20):
    for attempt in range(1, int(attempts) + 1):
        try:
            resp = session.post(url, data=json.dumps(payload), timeout=180)
        except requests.RequestException as e:
            if attempt >= attempts:
                raise
            sleep_sec = min(int(max_sleep), int(base_sleep) * attempt)
            print(f"  retry {label} attempt {attempt}/{attempts} after {sleep_sec}s ({e})")
            time.sleep(sleep_sec)
            continue

        if resp.status_code < 400:
            return

        err = f"HTTP {resp.status_code}: {resp.text[:800]}"
        retriable = resp.status_code >= 500 or is_retriable_write_error(resp.text)
        if retriable and attempt < attempts:
            sleep_sec = min(int(max_sleep), int(base_sleep) * attempt)
            print(f"  retry {label} attempt {attempt}/{attempts} after {sleep_sec}s ({err})")
            time.sleep(sleep_sec)
            continue
        _check(resp)


def vid(bank, acct):
    return f"{bank}:{acct}"


def normalize_headers(raw_headers):
    seen = {}
    normalized = []
    for h in (raw_headers or []):
        name = str(h or "").strip()
        idx = seen.get(name, 0)
        normalized.append(name if idx == 0 else f"{name}.{idx}")
        seen[name] = idx + 1
    return normalized


def row_to_dict(headers, row):
    vals = [str(v) for v in row]
    if len(vals) < len(headers):
        vals.extend([""] * (len(headers) - len(vals)))
    return {h: vals[i] for i, h in enumerate(headers)}


def to_float(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return 0.0


def to_int(x):
    s = str(x).strip()
    try:
        return int(float(s)) if s else 0
    except (TypeError, ValueError):
        return 0


def resolve_columns(headers):
    cols = {c.lower().strip(): c for c in headers}

    def col(*cands):
        for c in cands:
            if c.lower() in cols:
                return cols[c.lower()]
        raise KeyError(f"None of {cands} found in {headers}")

    return {
        "ts": col("Timestamp"),
        "fbank": col("From Bank"),
        "facct": col("Account"),
        "tbank": col("To Bank"),
        "tacct": col("Account.1", "Account 1"),
        "arecv": col("Amount Received"),
        "rcur": col("Receiving Currency"),
        "apaid": col("Amount Paid"),
        "pcur": col("Payment Currency"),
        "pfmt": col("Payment Format"),
        "flag": col("Is Laundering"),
    }


def make_edge(row, row_id, c):
    from_bank = str(row.get(c["fbank"], ""))
    from_acct = str(row.get(c["facct"], ""))
    to_bank = str(row.get(c["tbank"], ""))
    to_acct = str(row.get(c["tacct"], ""))
    return {
        "label": "transfer",
        "outV": vid(from_bank, from_acct), "outVLabel": "account",
        "inV": vid(to_bank, to_acct), "inVLabel": "account",
        "properties": {
            "ts": str(row.get(c["ts"], "")),
            "amount_paid": to_float(row.get(c["apaid"], "")),
            "amount_recv": to_float(row.get(c["arecv"], "")),
            "pay_currency": str(row.get(c["pcur"], "")),
            "recv_currency": str(row.get(c["rcur"], "")),
            "pay_format": str(row.get(c["pfmt"], "")),
            "is_laundering": to_int(row.get(c["flag"], "")),
            "row": int(row_id),
        },
    }


def graph_has_transfer_data():
    try:
        result = gremlin("g.E().hasLabel('transfer').limit(1).count()", eval_timeout_ms=20_000)
        return bool(result and int(result[0]) > 0)
    except Exception:
        return False


def clear_graph_data():
    print("Clearing graph data...")
    resp = session.delete(
        f"{BASE}/clear",
        params={"confirm_message": "I'm sure to delete all data"},
        timeout=120,
    )
    _check(resp)


def load_aml_transactions(max_rows=LOAD_ROWS):
    if not TRANS_CSV.exists():
        raise FileNotFoundError(f"Transactions CSV not found: {TRANS_CSV}")

    seen_accounts = set()
    vertex_batch = []
    edge_batch = []
    rows_loaded = 0
    vertices_loaded = 0
    edges_loaded = 0
    start_ts = time.time()

    with TRANS_CSV.open("r", newline="", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        headers = normalize_headers(next(reader, []))
        c = resolve_columns(headers)

        for row_id, raw_row in enumerate(reader):
            if max_rows is not None and rows_loaded >= int(max_rows):
                break

            row = row_to_dict(headers, raw_row)
            from_id = vid(str(row.get(c["fbank"], "")), str(row.get(c["facct"], "")))
            to_id = vid(str(row.get(c["tbank"], "")), str(row.get(c["tacct"], "")))

            if from_id not in seen_accounts:
                seen_accounts.add(from_id)
                vertex_batch.append({
                    "label": "account",
                    "id": from_id,
                    "properties": {
                        "bank": str(row.get(c["fbank"], "")),
                        "acct": str(row.get(c["facct"], "")),
                    },
                })

            if to_id not in seen_accounts:
                seen_accounts.add(to_id)
                vertex_batch.append({
                    "label": "account",
                    "id": to_id,
                    "properties": {
                        "bank": str(row.get(c["tbank"], "")),
                        "acct": str(row.get(c["tacct"], "")),
                    },
                })

            edge_batch.append(make_edge(row, row_id, c))

            if len(vertex_batch) >= int(VERTEX_BATCH):
                post_batch_with_retry(f"{BASE}/graph/vertices/batch", vertex_batch, "vertices")
                vertices_loaded += len(vertex_batch)
                vertex_batch.clear()

            if len(edge_batch) >= int(EDGE_BATCH):
                if vertex_batch:
                    post_batch_with_retry(f"{BASE}/graph/vertices/batch", vertex_batch, "vertices")
                    vertices_loaded += len(vertex_batch)
                    vertex_batch.clear()
                post_batch_with_retry(
                    f"{BASE}/graph/edges/batch?check_vertex=false",
                    edge_batch,
                    "edges",
                )
                edges_loaded += len(edge_batch)
                edge_batch.clear()

            rows_loaded += 1
            if rows_loaded % int(LOAD_PROGRESS_EVERY_ROWS) == 0:
                elapsed = max(time.time() - start_ts, 1e-9)
                print(
                    f"  rows={rows_loaded:,} inserted(v={vertices_loaded:,},e={edges_loaded:,}) "
                    f"avg_rows/s={rows_loaded / elapsed:,.1f}"
                )

            if MAX_SEEN_ACCOUNTS is not None and len(seen_accounts) >= int(MAX_SEEN_ACCOUNTS):
                seen_accounts.clear()

    if vertex_batch:
        post_batch_with_retry(f"{BASE}/graph/vertices/batch", vertex_batch, "vertices")
        vertices_loaded += len(vertex_batch)
    if edge_batch:
        post_batch_with_retry(f"{BASE}/graph/edges/batch?check_vertex=false", edge_batch, "edges")
        edges_loaded += len(edge_batch)

    elapsed = max(time.time() - start_ts, 1e-9)
    print(
        f"Loaded rows={rows_loaded:,} vertices={vertices_loaded:,} edges={edges_loaded:,} "
        f"in {elapsed:.1f}s"
    )


def maybe_load_data():
    if not LOAD_ENABLED:
        print("LOAD_ENABLED=False -> skipping schema/data load")
        return

    print("Ensuring AML schema...")
    try:
        ensure_aml_schema()
    except Exception as e:
        raise RuntimeError(
            "Schema setup failed fast. Check HugeGraph schema/task endpoints and logs. "
            f"Cause: {e}"
        ) from e
    print("AML schema is ready")

    if CLEAR_GRAPH_BEFORE_LOAD:
        clear_graph_data()

    has_data = graph_has_transfer_data()
    print("Existing transfer data:", has_data)

    should_load = FORCE_RELOAD or (LOAD_IF_GRAPH_EMPTY and not has_data)
    if not should_load:
        print("Skipping load (graph already has data; set FORCE_RELOAD=True to load again)")
        return

    if "ensure_aml_csvs" not in globals():
        raise RuntimeError(
            "CSV preflight helpers are not defined. "
            "Run Step 1.6 or execute the notebook from top to bottom."
        )

    ensure_aml_csvs(download_if_missing=KAGGLE_AUTO_DOWNLOAD)

    if not TRANS_CSV.exists():
        raise FileNotFoundError(
            f"Missing transactions CSV: {TRANS_CSV}. Place {TRANS_CSV_NAME} under {DATA_DIR}."
        )

    if ACCOUNTS_CSV.exists():
        print(f"Found accounts CSV (optional): {ACCOUNTS_CSV}")
    else:
        print(f"Accounts CSV not found (optional): {ACCOUNTS_CSV}")

    load_aml_transactions(max_rows=LOAD_ROWS)


maybe_load_data()


## Step 3 - Health check and dynamic seeds


In [ ]:
assert server_up(), "HugeGraph server is not reachable on :8080"
print("HugeGraph is up. Graphs:", session.get(f"http://{HG_HOST}:{HG_PORT}/graphs").json())
print("Pinned Gremlin alias:", GREMLIN_PINNED_ALIAS)
print("Gremlin sanity check 1+1:", gremlin("1+1"))

sample_accounts = show(
    "g.V().hasLabel('account').limit(3).project('id','bank','acct').by(id).by('bank').by('acct')",
    "Sample account vertices",
)
sample_transfers = show(
    "g.E().hasLabel('transfer').limit(3).project('id','from','to','amount','flag')"
    ".by(id).by(outV().id()).by(inV().id()).by('amount_paid').by('is_laundering')",
    "Sample transfer edges",
)

SEED_OUT = (gremlin("g.V().hasLabel('account').where(outE('transfer')).id().limit(1)") or [None])[0]
SEED_IN = (gremlin("g.V().hasLabel('account').where(inE('transfer')).id().limit(1)") or [None])[0]
SEED_ANY = (gremlin("g.V().hasLabel('account').id().limit(1)") or [None])[0]
FLAGGED_SEED = (
    gremlin("g.E().hasLabel('transfer').has('is_laundering',1).outV().id().limit(1)") or [None]
)[0]

SEED = SEED_OUT or SEED_IN or SEED_ANY
print("SEED_OUT:", SEED_OUT)
print("SEED_IN:", SEED_IN)
print("SEED_ANY:", SEED_ANY)
print("FLAGGED_SEED:", FLAGGED_SEED)
print("Active seed:", SEED)

SEED_BANK = None
if SEED is not None:
    _seed_vertex = gremlin(
        "g.V(s).project('id','bank','acct').by(id).by('bank').by('acct')",
        bindings={"s": SEED}
    )
    if _seed_vertex:
        SEED_BANK = _seed_vertex[0].get("bank")
        print("Seed vertex:", _seed_vertex[0])


## 1 - Root steps (`g.V()`, `g.E()`, `g.V(id)`)


In [ ]:
show(
    "g.V().count()"
)
show(
    "g.E().count()"
)
show(
    f"g.V().hasLabel('account').limit({int(ROOT_COUNT_SCOPE_LIMIT)}).count()",
    "1a Root: scoped account count",
)
show(
    f"g.E().hasLabel('transfer').limit({int(ROOT_COUNT_SCOPE_LIMIT)}).count()",
    "1b Root: scoped transfer count",
)
if SEED is not None:
    show(
        "g.V(s).project('id','bank','acct').by(id).by('bank').by('acct')",
        "1c Root: g.V(id)",
        bindings={"s": SEED},
    )


## 2 - Filter steps (`has`, `hasLabel`, `hasId`, `hasNot`, `is`)


In [ ]:
if SEED_BANK is not None:
    show(
        "g.V().hasLabel('account').has('bank', b).limit(10)"
        ".project('id','bank','acct').by(id).by('bank').by('acct')",
        "2a has + hasLabel",
        bindings={"b": SEED_BANK},
    )

show(
    "g.E().hasLabel('transfer').has('is_laundering',1).limit(10)"
    ".project('from','to','amount','currency').by(outV().id()).by(inV().id())"
    ".by('amount_paid').by('pay_currency')",
    "2b has on edge property",
)

if SEED is not None:
    show(
        "g.V().hasId(s).project('id','bank','acct').by(id).by('bank').by('acct')",
        "2c hasId",
        bindings={"s": SEED},
    )

show(
    f"g.V().hasLabel('account').limit({int(HASNOT_SCOPE_LIMIT)}).hasNot('bank').count()",
    "2d hasNot on scoped subset",
)

if SEED is not None:
    show(
        "g.V(s).outE('transfer').values('amount_paid').is(gt(0)).limit(10)",
        "2e is(gt(...))",
        bindings={"s": SEED},
    )


## 3 - Hop steps (`out`, `in`, `both`, `outE`, `inE`, `bothE`, `outV`, `inV`, `bothV`, `otherV`)


In [ ]:
if SEED is not None:
    show(
        "g.V(s).out('transfer').limit(10).project('id','bank','acct').by(id).by('bank').by('acct')",
        "3a out('transfer')",
        bindings={"s": SEED},
    )

if SEED is not None:
    show(
        "g.V(s).in('transfer').limit(10).project('id','bank','acct').by(id).by('bank').by('acct')",
        "3b in('transfer')",
        bindings={"s": SEED},
    )

if SEED is not None:
    show(
        "g.V(s).both('transfer').dedup().limit(10).project('id','bank','acct')"
        ".by(id).by('bank').by('acct')",
        "3c both('transfer')",
        bindings={"s": SEED},
    )

if SEED is not None:
    show(
        "g.V(s).outE('transfer').limit(10).project('id','amount','currency','flag')"
        ".by(id).by('amount_paid').by('pay_currency').by('is_laundering')",
        "3d outE('transfer')",
        bindings={"s": SEED},
    )

if SEED is not None:
    show(
        "g.V(s).inE('transfer').limit(10).project('id','amount','currency','flag')"
        ".by(id).by('amount_paid').by('pay_currency').by('is_laundering')",
        "3e inE('transfer')",
        bindings={"s": SEED},
    )

if SEED is not None:
    show(
        "g.V(s).bothE('transfer').limit(10).project('id','amount').by(id).by('amount_paid')",
        "3f bothE('transfer')",
        bindings={"s": SEED},
    )

show(
    "g.E().hasLabel('transfer').limit(10).outV().project('id','bank','acct').by(id).by('bank').by('acct')",
    "3g outV() from edge root",
)

show(
    "g.E().hasLabel('transfer').limit(10).inV().project('id','bank','acct').by(id).by('bank').by('acct')",
    "3h inV() from edge root",
)

show(
    "g.E().hasLabel('transfer').limit(10).bothV().dedup().project('id','bank','acct')"
    ".by(id).by('bank').by('acct')",
    "3i bothV() from edge root",
)

if SEED is not None:
    show(
        "g.V(s).bothE('transfer').limit(10).otherV().project('id','bank','acct')"
        ".by(id).by('bank').by('acct')",
        "3j otherV()",
        bindings={"s": SEED},
    )


## 4 - Repeat / path steps


In [ ]:
if SEED is not None:
    show(
        "g.V(s).repeat(out('transfer').simplePath()).times(2).path().by(id).limit(10)",
        "4a 2-hop out simplePath",
        bindings={"s": SEED},
    )

if SEED is not None:
    show(
        "g.V(s).repeat(both('transfer').simplePath()).times(2).path().by(id).limit(10)",
        "4b 2-hop both simplePath",
        bindings={"s": SEED},
    )


## 5 - Aggregation (`count`, `sum`, `mean`, `groupCount`)


In [ ]:
if SEED is not None:
    show("g.V(s).outE('transfer').count()", "5a out count", bindings={"s": SEED})
    show("g.V(s).inE('transfer').count()", "5b in count", bindings={"s": SEED})
    show("g.V(s).outE('transfer').values('amount_paid').sum()", "5c sum(amount_paid)",
         bindings={"s": SEED})
    show("g.V(s).outE('transfer').values('amount_paid').mean()", "5d mean(amount_paid)",
         bindings={"s": SEED})

show(
    f"g.V().hasLabel('account').limit({int(GROUPCOUNT_ACCOUNT_SCOPE_LIMIT)}).outE('transfer').limit({int(GROUPCOUNT_EDGE_SCOPE_LIMIT)})"
    ".groupCount().by('pay_currency').unfold()"
    ".project('currency','count')"
    ".by(select(org.apache.tinkerpop.gremlin.structure.Column.keys))"
    ".by(select(org.apache.tinkerpop.gremlin.structure.Column.values))"
    ".order().by(select('count'), desc).limit(10)",
    "5e groupCount by pay_currency (scoped)",
)


## 6 - Projection (`values`, `project().by()`, `valueMap`, `identity`)


In [ ]:
show("g.V().hasLabel('account').values('acct').limit(10)", "6a values('acct')")

show(
    "g.V().hasLabel('account').limit(10).project('id','bank','acct').by(id).by('bank').by('acct')",
    "6b project().by() on vertices",
)

show("g.V().hasLabel('account').valueMap('bank','acct').limit(10)", "6c valueMap on vertices")

show(
    "g.E().hasLabel('transfer').valueMap('amount_paid','pay_currency','is_laundering').limit(10)",
    "6d valueMap on edges",
)

show(
    "g.V().hasLabel('account').limit(20).project('account','out_deg','in_deg')"
    ".by(id).by(outE('transfer').count()).by(inE('transfer').count())"
    ".order().by(select('out_deg'), desc).limit(10)",
    "6e project with nested counts",
)


## 7 - Ordering and paging (`order().by`, `limit`, `range`, `dedup`)


In [ ]:
show(
    f"g.V().hasLabel('account').limit({int(RANGE_SCOPE_LIMIT)}).project('account','out_deg')"
    ".by(id).by(outE('transfer').count())"
    ".order().by(select('out_deg'), desc).limit(10)",
    "7a order by projected out degree",
)

show(
    f"g.V().hasLabel('account').limit({int(RANGE_SCOPE_LIMIT)}).order().by(id).range(0,10).id()",
    "7b page 1 via range(0,10) on scoped subset",
)

show(
    f"g.V().hasLabel('account').limit({int(RANGE_SCOPE_LIMIT)}).order().by(id).range(10,20).id()",
    "7c page 2 via range(10,20) on scoped subset",
)

show(
    f"g.V().hasLabel('account').limit({int(DEDUP_SCOPE_LIMIT)}).both('transfer').dedup().count()",
    "7d dedup over neighbors (scoped)",
)


## 8 - Alias / select / where


In [ ]:
show(
    f"g.V().hasLabel('account').limit({int(ALIAS_PAIR_SCOPE_LIMIT)}).as('a').out('transfer').as('b')"
    ".where('a', neq('b')).select('a','b').by(id).by(id).limit(10)",
    "8a as + select + where(neq)",
)

show(
    f"g.V().hasLabel('account').limit({int(FLAGGED_PROJECT_SCOPE_LIMIT)}).project('account','flagged_out')"
    ".by(id).by(outE('transfer').has('is_laundering',1).count())"
    ".where(select('flagged_out').is(gt(0)))"
    ".order().by(select('flagged_out'), desc).limit(10)",
    "8b where(select(...).is(gt(...)))",
)


## 9 - Compound `where` (`and`, `or`, `not`)


In [ ]:
show(
    f"g.V().hasLabel('account').limit({int(COMPOUND_WHERE_SCOPE_LIMIT)})"
    ".where(__.and(__.outE('transfer').count().is(gt(0)), __.inE('transfer').count().is(gt(0))))"
    ".project('account','in_deg','out_deg')"
    ".by(id).by(inE('transfer').count()).by(outE('transfer').count())"
    ".limit(10)",
    "9a where(__.and(...))",
)

show(
    f"g.V().hasLabel('account').limit({int(COMPOUND_WHERE_SCOPE_LIMIT)})"
    ".where(__.or(__.outE('transfer').has('is_laundering',1), __.inE('transfer').has('is_laundering',1)))"
    ".project('account','flagged_in','flagged_out')"
    ".by(id)"
    ".by(inE('transfer').has('is_laundering',1).count())"
    ".by(outE('transfer').has('is_laundering',1).count())"
    ".limit(10)",
    "9b where(__.or(...))",
)

show(
    f"g.V().hasLabel('account').limit({int(COMPOUND_WHERE_SCOPE_LIMIT)}).where(__.not(__.outE('transfer'))).id().limit(10)",
    "9c where(__.not(...))",
)


## 10 - `groupCount` with alias key


In [ ]:
show(
    f"g.V().hasLabel('account').limit({int(BANK_GROUP_SCOPE_LIMIT)}).as('a').groupCount()"
    ".by(select('a').by('bank')).unfold()"
    ".project('bank','accounts')"
    ".by(select(org.apache.tinkerpop.gremlin.structure.Column.keys))"
    ".by(select(org.apache.tinkerpop.gremlin.structure.Column.values))"
    ".order().by(select('accounts'), desc).limit(10)",
    "10a bank-level account counts",
)


## 11 - Edge-root projection with `outV` / `inV`


In [ ]:
show(
    "g.E().hasLabel('transfer').limit(15)"
    ".project('from','to','amount','currency','format','flag','ts')"
    ".by(outV().id()).by(inV().id())"
    ".by('amount_paid').by('pay_currency').by('pay_format')"
    ".by('is_laundering').by('ts')",
    "11a edge-root projection",
)


## 12 - `simplePath` across multi-hop traversals


In [ ]:
if SEED is not None:
    show(
        "g.V(s).repeat(out('transfer').simplePath()).times(3).path().by(id).limit(10)",
        "12a 3-hop simplePath",
        bindings={"s": SEED},
    )


## 13 - `identity()` as no-op and in projection


In [ ]:
show(
    "g.V().hasLabel('account').limit(5).identity().project('id','bank').by(id).by('bank')",
    "13a identity() no-op",
)

show(
    "g.V().hasLabel('account').limit(5).project('id','self').by(id).by(identity())",
    "13b by(identity())",
)


## 14 - Bounded account-activity ranking query

This is a bounded account-activity ranking query based on the `is_laundering` edge flag.


In [ ]:
show(
    f"g.V().hasLabel('account').limit({int(RANKING_SCOPE_LIMIT)})"
    ".where(outE('transfer').has('is_laundering',1))"
    ".project('account','bank','flagged_out','total_out','fan_out')"
    ".by(id).by('bank')"
    ".by(outE('transfer').has('is_laundering',1).count())"
    ".by(outE('transfer').count())"
    ".by(out('transfer').dedup().count())"
    ".order().by(select('flagged_out'), desc).limit(20)",
    "14a account ranking by flagged outgoing transfers",
)


## 15 - Deep multi-hop traversals (bounded for HugeGraph)

These cells are tuned for runtime safety:
- use a single seed
- keep strict limits
- avoid unbounded global edge scans


In [ ]:
HUB = FLAGGED_SEED or SEED
print("Deep-traversal HUB:", HUB)

if HUB is not None:
    show(
        "g.V(h).repeat(out('transfer').simplePath()).times(5).path().by(id).limit(10)",
        "15a 5-hop path sample",
        bindings={"h": HUB},
    )

if HUB is not None:
    show(
        f"g.V(h).repeat(out('transfer').limit({int(HOP8_OUT_LIMIT)})).times(8).dedup().limit({int(HOP8_RESULT_LIMIT)}).count()",
        "15b distinct accounts reachable in 8 hops (scoped)",
        bindings={"h": HUB},
    )

seed_rows = show(
    "g.E().hasLabel('transfer').has('is_laundering',1)"
    ".groupCount().by(outV().id()).unfold()"
    ".project('seed','flagged_out_edges')"
    ".by(select(org.apache.tinkerpop.gremlin.structure.Column.keys))"
    ".by(select(org.apache.tinkerpop.gremlin.structure.Column.values))"
    ".order().by(select('flagged_out_edges'), desc).limit(5)",
    "15c accounts with highest flagged outgoing transfers",
)

if seed_rows:
    print("\n15d 6-hop reachability per seed")
    seed_scoped_query = (
        f"g.V(s).repeat(out('transfer').limit({int(HOP6_OUT_LIMIT)})).times(6)"
        f".dedup().limit({int(HOP6_RESULT_LIMIT)}).count()"
    )
    for row in seed_rows:
        s = row.get("seed")
        try:
            c = gremlin(seed_scoped_query, bindings={"s": s}, eval_timeout_ms=120_000)
            cnt = c[0] if c else 0
            print(
                f"  seed={s}  flagged_out_edges={row.get('flagged_out_edges')} "
                f"6hop_scoped={cnt}"
            )
        except Exception as e:
            err_text = str(e)
            if needs_alias_fallback(err_text):
                try:
                    wait_for_gremlin_alias_binding(
                        timeout_sec=40,
                        poll_sec=2,
                        required_consecutive=2,
                    )
                    c = gremlin(seed_scoped_query, bindings={"s": s}, eval_timeout_ms=120_000)
                    cnt = c[0] if c else 0
                    print(
                        f"  seed={s}  flagged_out_edges={row.get('flagged_out_edges')} "
                        f"6hop_scoped={cnt} (after alias rebind)"
                    )
                    continue
                except Exception as e2:
                    print(f"  seed={s} FAILED (scoped after alias rebind): {e2}")
                    continue
            print(f"  seed={s} FAILED (scoped): {e}")
